In [71]:
import numpy as np
import pyccl as ccl
from astropy.io import fits
import matplotlib.pyplot as plt

dataset = "/home/weichen/cosmo_practice/dataset_hsc_y3.fits"
with fits.open(dataset) as hd:
    ds_t, wp_t  = hd['ds'].data, hd['wp'].data
    xip_t, xim_t = hd['xip'].data, hd['xim'].data
    full_cov = hd['COVMAT'].data
    z_s, nz_s = hd['nz_source'].data['Z_MID'], hd['nz_source'].data['BIN1']
    nzl = hd['nz_lens'].data
    z_lg = nzl['Z_MID']
    nz_l = [nzl['BIN1'], nzl['BIN2'], nzl['BIN3']]   # LOWZ, CMASS1, CMASS2 的 p_l(z)

    print(repr(hd['ds'].header))
    print(hd['ds'].columns)

alpha_mag = [2.259, 3.563, 3.729]   # Table I 的 prior 中心值
SIGC_CONST = 1.6624e18              # c²/(4πG) [M⊙/Mpc], Eq.(8) 的常數
ds_rmax = {1: 30.0, 2: 40.0, 3: 80.0}

ds_cut = np.zeros(len(ds_t), dtype=bool)
for _b, _rmax in ds_rmax.items():
    ds_cut |= (ds_t['BIN1'] == _b) & (ds_t['ANG'] >= 12.0) & (ds_t['ANG'] <= _rmax)

wp_cut  = (wp_t['ANG'] >= 8.0)  & (wp_t['ANG'] <= 80.0)
xip_cut = (xip_t['ANG'] >= 7.9)  & (xip_t['ANG'] <= 50.1)
xim_cut = (xim_t['ANG'] >= 31.6) & (xim_t['ANG'] <= 158.0)

n_ds_full, n_xip_full, n_xim_full, n_wp_full = len(ds_t), len(xip_t), len(xim_t), len(wp_t)
off_xip = n_ds_full
off_xim = n_ds_full + n_xip_full
off_wp = n_ds_full + n_xip_full + n_xim_full   # 全長

keep = np.concatenate([np.where(ds_cut)[0],
                       np.where(xip_cut)[0] + off_xip,
                       np.where(xim_cut)[0] + off_xim,
                       np.where(wp_cut)[0]  + off_wp])

cov = full_cov[np.ix_(keep, keep)]
icov = np.linalg.inv(cov) * (107*13 - 74 - 2) / (107*13 - 1)   # Hartlap
err = np.sqrt(np.diag(cov))   # error bar

data_vec = np.concatenate([ds_t['VALUE'][ds_cut], xip_t['VALUE'][xip_cut],
                           xim_t['VALUE'][xim_cut], wp_t['VALUE'][wp_cut]])

n_ds, n_xip, n_xim, n_wp = ds_cut.sum(), xip_cut.sum(), xim_cut.sum(), wp_cut.sum()

print(f"data vector: ds={n_ds}, xip={n_xip}, xim={n_xim}, wp={n_wp}, total={len(data_vec)}")

z_l_eff = [0.2607, 0.5106, 0.6264]


XTENSION= 'BINTABLE'           / binary table extension                         
BITPIX  =                    8 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                   56 / length of dimension 1                          
NAXIS2  =                   90 / length of dimension 2                          
PCOUNT  =                    0 / number of group parameters                     
GCOUNT  =                    1 / number of groups                               
TFIELDS =                    7 / number of table fields                         
2PTDATA =                    T                                                  
EXTNAME = 'ds      '                                                            
QUANT1  = 'GPR     '                                                            
QUANT2  = 'G+R     '                                                            
KERNEL_1= 'nz_lens '        

In [72]:
om_m_ref = 0.279
C_OVER_H0 = 2997.92458

def E_flat(z, Om):
    return np.sqrt(Om*(1.0+z)**3 + (1.0-Om))

def chi_hinv_flat(z, Om, n=512):
    zz = np.linspace(0.0, z, n)
    return C_OVER_H0 * np.trapezoid(1.0/E_flat(zz, Om), zz)

def sigcrit_inv_mean(zl, om, z_s, nz_s):
    """Source-n(z)-averaged comoving Sigma_crit^-1 (constants cancel in the ratio)."""
    Dl = chi_hinv_flat(zl, om)
    Ds = np.array([chi_hinv_flat(z, om) for z in z_s])
    integ = np.where(z_s > zl, Dl*(Ds - Dl)/np.where(Ds > 0, Ds, 1.0), 0.0) * nz_s
    return np.trapezoid(integ, z_s) / np.trapezoid(nz_s, z_s)

In [73]:
def detg_alpha(a, om, beta_detg, p_detg):
    # P_DETG(k,a) / P_LCDM(k,a)

    a = np.asarray(a)
    E2 = om / a**3 + (1.0 - om)
    omega_de_ratio = 1.0 / E2

    return 1.0 - beta_detg * omega_de_ratio**p_detg


def detg_growth_rate(cosmo, a, om, beta_detg, p_detg):
    """
    f_DETG = d ln(D_DETG) / d ln(a)
    D_DETG = D_LCDM * sqrt(alpha)
    """
    E2 = om / a**3 + (1.0 - om)
    omega_de_ratio = 1.0 / E2
    omega_m_a = (om / a**3) / E2

    alpha = detg_alpha(a, om, beta_detg, p_detg)

    return (ccl.growth_rate(cosmo, a) - 1.5 * beta_detg * p_detg * omega_m_a * omega_de_ratio**p_detg / alpha)


def make_detg_pk2d(cosmo, om, beta_detg, p_detg):
    # 一階近似: P_NL_DETG(k,a) = alpha(a) * P_NL_LCDM(k,a)

    pk_lcdm = cosmo.get_nonlin_power()
    a_arr, lk_arr, pk_arr = pk_lcdm.get_spline_arrays()

    alpha = detg_alpha(a_arr, om, beta_detg, p_detg)
    if np.any(alpha <= 0):
        raise ValueError(
            "DETG gives alpha(a) <= 0. "
            "Check beta_DETG and p_DETG."
        )

    return ccl.Pk2D( 
        a_arr=a_arr, lk_arr=lk_arr, 
        pk_arr=np.log(pk_arr) + np.log(alpha)[:, None],
        is_logp=True, extrap_order_lok=1, extrap_order_hik=2
    )

In [74]:
def theory_vector(cosmo, beta_detg, b, p_detg=3.0, A_IA=0, dm=0, dzph=0, pi_max=100):
    h = cosmo['h']
    om = cosmo['Omega_c'] + cosmo['Omega_b']
    pk_detg = make_detg_pk2d(cosmo=cosmo, om=om, beta_detg=beta_detg, p_detg=p_detg)
    rho_m = ccl.rho_x(cosmo, 1.0, 'matter', is_comoving=True)
    m_ds, m_wp = [], []
    nz_true = np.interp(z_s + dzph, z_s, nz_s, left=0., right=0.)

    for i in range(3):
        zl = z_l_eff[i]
        a_l = 1.0/(1.0+z_l_eff[i])
        
        E_C, E_ref = E_flat(zl, om), E_flat(zl, om_m_ref)
        chi_C, chi_ref = chi_hinv_flat(zl, om), chi_hinv_flat(zl, om_m_ref)
        f_ds = (sigcrit_inv_mean(zl, om, z_s, nz_true) / sigcrit_inv_mean(zl, om_m_ref, z_s, nz_s)) 
        
        R_fac, E_fac = chi_C/chi_ref, E_C/E_ref
        pimax_wp = (E_ref/E_C)*pi_max

        # ===============================
        rp_ds = ds_t['ANG'][(ds_t['BIN1']==i+1) & ds_cut]
        rp_wp = wp_t['ANG'][(wp_t['BIN1']==i+1) & wp_cut]
        rp_ds_true = R_fac * rp_ds
        rp_wp_true = R_fac * rp_wp

        # ============ ΔΣ_mag:Eq.(4) ============
        kern_Mpc = sigcrit_inv_mean(zl, om, z_s, nz_true) / h
        Sig_c = SIGC_CONST / ((1.0 + zl) * kern_Mpc)  # M⊙/Mpc²

        # --- Cκ(ℓ; zl, zs)(Eq. 9–10)---
        tr_l = ccl.WeakLensingTracer(cosmo, dndz=(z_lg, nz_l[i]))   # lens face's κ kernel
        tr_s2 = ccl.WeakLensingTracer(cosmo, dndz=(z_s, nz_true))    # 平移後的 source
        ell_m = np.geomspace(0.1, 1e5, 512)
        cl_ls = ccl.angular_cl(cosmo, tr_l, tr_s2, ell=ell_m, p_of_k_a=pk_detg,)

        # --- ∫ ℓdℓ/2π Cκ J2(ℓR/χl)(Eq. 7 Hankel intergrate, CCL 'GL' -> J2)---
        theta_deg = (rp_ds_true / chi_C) * (180.0/np.pi)   # θ=R/χl
        gt = ccl.correlation(cosmo, ell=ell_m, C_ell=cl_ls,
                             theta=theta_deg, type='NG', method='FFTLog')

        # --- Eq. 7 ---
        ds_mag = 2.0*(alpha_mag[i]-1.0) * Sig_c * gt / (h*1e12)

        # ===============================
        pi_ds = np.linspace(0.0, pi_max, 300)
        pi_wp = np.linspace(0.0, pimax_wp, 300)    # wp turn to Πmax
        
        rmax_all = max(np.max(ds_t["ANG"]), np.max(wp_t["ANG"]))
        r_grid = np.geomspace(1e-3, rmax_all*20 + 100, 1000)

        f_detg = detg_growth_rate(cosmo=cosmo, a=a_l, om=om, beta_detg=beta_detg, p_detg=p_detg,)
        beta_rsd = f_detg / b[i]

        xi_mm = ccl.correlation_3d(cosmo, a=a_l, r=r_grid/h, p_of_k_a=pk_detg)
        xi_gm = b[i] * xi_mm
        xi_gg = b[i]**2 * xi_mm

        # J3, J5 (RSD)
        J3 = np.zeros_like(r_grid); J5 = np.zeros_like(r_grid)
        for j, r in enumerate(r_grid):
            mk = r_grid <= r
            if mk.sum() < 2: continue
            J3[j] = np.trapezoid(xi_gg[mk]*r_grid[mk]**2, r_grid[mk])/r**3
            J5[j] = np.trapezoid(xi_gg[mk]*r_grid[mk]**4, r_grid[mk])/r**5

        def ds_at(rp):
            r3d = np.sqrt(rp**2 + pi_ds**2)
            Sig = 2*rho_m*np.trapezoid(np.interp(r3d, r_grid, xi_gm), pi_ds)/(h**2*1e12)
            rin = np.linspace(1e-3, rp, 100)
            r3i = np.sqrt(rin[None,:]**2 + pi_ds[:,None]**2)
            Sin = 2*rho_m*np.trapezoid(np.interp(r3i, r_grid, xi_gm), pi_ds, axis=0)/(h**2*1e12)
            
            return 2/rp**2*np.trapezoid(Sin*rin, rin) - Sig

        def wp_at(rp):
            r3d = np.sqrt(rp**2 + pi_wp**2); mu = pi_wp/r3d
            xl = np.interp(r3d, r_grid, xi_gg)
            j3 = np.interp(r3d, r_grid, J3); j5 = np.interp(r3d, r_grid, J5)
            x0 = (1 + 2/3*beta_rsd + beta_rsd**2/5)*xl
            x2 = (4/3*beta_rsd + 4/7*beta_rsd**2)*(xl - 3*j3)
            x4 = 8/35*beta_rsd**2*(xl + 7.5*j3 - 17.5*j5)
            L2 = .5*(3*mu**2-1); L4 = .125*(35*mu**4-30*mu**2+3)
            
            return 2*np.trapezoid(x0 + x2*L2 + x4*L4, pi_wp)

        m_ds += [(1.0 + dm)*f_ds*(ds_at(rp) + dsm)  for rp, dsm in zip(rp_ds_true, ds_mag)]
        m_wp += [E_fac * wp_at(rp) for rp in rp_wp_true]

    # cosmic shear = GG + GI + II (NLA intrinsic alignment) w/ (1+dm)^2 shear calib.
    ell = np.geomspace(0.1, 1e5, 1024)     # wide range -> no FFTLog edge drop
    ia = (z_s, np.full_like(z_s, float(A_IA)))        # NLA amplitude A_IA(z)=const
    tr = ccl.WeakLensingTracer(cosmo, dndz=(z_s, nz_true), ia_bias=ia)  # CCL applies -C1 rho_c Om/D
    cl = ccl.angular_cl(cosmo, tr, tr, ell=ell, p_of_k_a=pk_detg)
    fac_m = (1.0 + dm)**2
    xip = fac_m*ccl.correlation(cosmo, ell=ell, C_ell=cl, 
                                theta=xip_t['ANG'][xip_cut]/60., type='GG+', method='FFTLog')
    xim = fac_m*ccl.correlation(cosmo, ell=ell, C_ell=cl, 
                                theta=xim_t['ANG'][xim_cut]/60., type='GG-', method='FFTLog')
        
    return np.concatenate([m_ds, np.atleast_1d(xip), np.atleast_1d(xim), m_wp])

def make_cosmo(ombh2, omch2, H0, ns, sigma8):
    h = H0/100
    return ccl.Cosmology(Omega_c=omch2/h**2, Omega_b=ombh2/h**2, h=h,
                         sigma8=sigma8, n_s=ns, matter_power_spectrum='halofit')

def chi2_of(theory):
    d = data_vec - theory
    return d @ icov @ d


In [75]:
# from Sugiyama et al. 2023
# background

# FN = "/home/weichen/cosmo_practice/final_y3_chains/hsc_y3_3x2pt_large_scale.txt"
# PARAM = "/home/weichen/cosmo_practice/final_y3_chains/hsc_y3_3x2pt_large_scale_paramnames.txt"

# with open(PARAM) as f:
#     hdr = [line.split()[0] for line in f if line.strip()]
# hdr += ["weight"]
# print(f"number of total field: {len(hdr)}")

# raw = np.loadtxt(FN)
# print('raw shape:', raw.shape, ' len(hdr):', len(hdr))
# col = {n: i for i, n in enumerate(hdr)}
# best = raw[np.argmax(raw[:, col["lnpost"]])]

# sig_names = [n for n in hdr if n.startswith('signal_')]
# sig = np.array([best[col[n]] for n in sig_names])
# print(sig_names)
# sig_ds = sig[:17]
# sig_xip = sig[17:25]
# sig_xim = sig[25:32]
# sig_wp = sig[32:]

# omb, omc = best[col['Ombh2']], best[col['Omch2']]
# omm, s8 = best[col['Omm']],   best[col['sigma8']]
# ns_ = best[col['ns']]
# om_m_ref = 0.279
# b_lit = [best[col[f'b1_{i}']] for i in range(3)]
# A_IA_lit = best[col['AIA']]  if 'AIA'  in col else 0.0     # NLA amplitude
# dm_lit = best[col['dm_0']] if 'dm_0' in col else 0.0     # residual multiplicative shear bias
# print(f"A_IA={A_IA_lit:.3f}, dm={dm_lit:.4f}")

# h_a = np.sqrt((omb + omc) / omm)  # 若 Omm 不含微中子
# h_b = np.sqrt((omb + omc + 0.00064) / omm)   # 若含 Mν=0.06eV(ωbased ≈0.00064)
# print(f"h(no ν)={h_a:.4f}  h(have ν)={h_b:.4f} → {abs(h_a-h_b)/h_a*100:.2f}% else")
# h_lit = h_a

# print(f"\nreference best-fit: Omm={omm:.4f} sigma8={s8:.4f} S8={best[col['S8']]:.4f}")
# print(f"b = {np.round(b_lit,3)}, lnpost = {best[col['lnpost']]:.2f}")

# plt.figure(figsize=(11,4))
# plt.semilogy(np.abs(sig), 'o-', ms=4)
# plt.xlabel('signal index'); plt.ylabel('|value| (log)')
# plt.title('HSC Y3 3×2pt Data Vector')
# plt.grid(alpha=.3); plt.savefig('signal_blocks.png', dpi=130)


In [76]:
# my theory data vector

from cobaya.model import get_model
from cobaya.yaml import yaml_load_file

info = yaml_load_file("/home/weichen/cosmo_practice/configs/0504_test_hsc_lens.yaml")
info.pop("output", None)
info.pop("sampler", None)
model = get_model(info)
lik = model.likelihood['test_hsc_lens.HSC_Lens']

p = dict(tau=0.0428, ombh2=0.02267, omch2=0.11616, H0=69.06, ns=0.9715,
         AIA=-0.79, dm_0=0.0, dpz_0=-0.03, logA=3.0098,
         X1=1.0, X2=1.0, X3=1.0, b1=1.86, b2=2.05, b3=2.02, A_planck=1.0)

cosmo_ccl = make_cosmo(0.02237, 0.1200, 67.36, 0.9649, 0.8111)
m_old = lik.get_theory_prediction(cosmo_ccl, **p)
m_new = theory_vector(cosmo_ccl, beta_detg=0.0, b=[1.86,2.05,2.02],
                      p_detg=3.0, A_IA=-0.79, dm=0.0, dzph=-0.03)

for beta in [0.0, 0.1, 0.2, 0.3, 0.5]:
    m = theory_vector(cosmo_ccl, beta_detg=beta, b=[1.86, 2.05, 2.02],
                      p_detg=3.0, A_IA=-0.79, dm=0.0, dzph=-0.03)
    print(f'beta={beta:.2f}  chi2={chi2_of(m):.2f}')

[camb] `camb` module loaded successfully from /home/weichen/cosmo_practice/.venv_wsl/lib/python3.12/site-packages/camb
[planck_2018_highl_plik.ttteee_lite] `clipy` module loaded successfully from /home/weichen/cosmo_practice/packages/cobaya_packages/code/planck/clipy/clipy
----
clipy_0.15
Checking likelihood '/home/weichen/cosmo_practice/packages/cobaya_packages/data/planck_2018/baseline/plc_3.0/hi_l/plik_lite/plik_lite_v22_TTTEEE.clik' on test data. got -292.286 expected -292.286 (diff -3.86217e-09)
----
before crop : TT 30 -> 2508, TE 30 -> 1996, EE 30 -> 1996
after crop  : TT 30 -> 2508, TE 30 -> 1996, EE 30 -> 1996
[test_hsc_lens.hsc_lens] Loading HSC 3x2pt Lens data: /home/weichen/cosmo_practice/dataset_hsc_y3.fits
[test_hsc_lens.hsc_lens] After scale cut: ds=17
xip=8
xim=7
wp=42
total=74
beta=0.00  chi2=215.16
beta=0.10  chi2=226.44
beta=0.20  chi2=241.37
beta=0.30  chi2=259.91
beta=0.50  chi2=307.85


In [77]:
r = m_new / m_old
d = (m_new - m_old) / err
for name, sl in [('ds', slice(0,n_ds)),
                 ('xip', slice(n_ds, n_ds+n_xip)),
                 ('xim', slice(n_ds+n_xip, n_ds+n_xip+n_xim)),
                 ('wp', slice(-n_wp, None))]:
    print(f'{name:4s} ratio max|r-1| = {np.abs(r[sl]-1).max():.3e}   '
          f'max|Δ/σ| = {np.abs(d[sl]).max():.3e}')

ds   ratio max|r-1| = 0.000e+00   max|Δ/σ| = 0.000e+00
xip  ratio max|r-1| = 0.000e+00   max|Δ/σ| = 0.000e+00
xim  ratio max|r-1| = 0.000e+00   max|Δ/σ| = 0.000e+00
wp   ratio max|r-1| = 0.000e+00   max|Δ/σ| = 0.000e+00


In [78]:
om_t = cosmo_ccl['Omega_c'] + cosmo_ccl['Omega_b']
pk0  = make_detg_pk2d(cosmo_ccl, om_t, 0.0, 3.0)
pkref = cosmo_ccl.get_nonlin_power()

kk = np.geomspace(1e-3, 10, 30)
for a_t in [1.0, 0.8, 0.6]:
    ratio = pk0(kk, a_t) / pkref(kk, a_t)
    print(f'a={a_t}: ratio range {ratio.min():.6f} – {ratio.max():.6f}')

kk_ext = np.geomspace(1e-5, 1e3, 40)
print((pk0(kk_ext, 0.8) / pkref(kk_ext, 0.8)).min(),
      (pk0(kk_ext, 0.8) / pkref(kk_ext, 0.8)).max())

a=1.0: ratio range 1.000000 – 1.000000
a=0.8: ratio range 1.000000 – 1.000000
a=0.6: ratio range 1.000000 – 1.000000
1.0 1.0
